In [ ]:
import json
import numpy as np
from tqdm import tqdm
import nltk
nltk.download("punkt")

# 載入 ragtruth 的路徑
SOURCE_FILE = "/kaggle/input/ragtruthful/source_info.jsonl"
RESPONSE_FILE = "/kaggle/input/ragtruthful/response.jsonl"

# 輸出最後處理完成的文件路徑
OUTPUT_SOURCE_FILE = "./source_info_with_chunks.jsonl"
OUTPUT_RESPONSE_FILE = "./response_with_chunks.jsonl"

In [ ]:
# =====================================
#   Read Source Info
# =====================================
source_info = {}
with open(SOURCE_FILE, "r") as f:
    for line in f:
        obj = json.loads(line)
        source_info[obj["source_id"]] = obj

# 統計容器
stats = {
    "qa": {
        "train": {"count": 0, "prompt_len": [], "source_len": [], "resp_len": [], "has_label": 0},
        "test":  {"count": 0, "prompt_len": [], "source_len": [], "resp_len": [], "has_label": 0}
    }
}

# =====================================
#   Read Responses
# =====================================
with open(RESPONSE_FILE, "r") as f:
    for line in f:
        obj = json.loads(line)

        sid = obj["source_id"]
        split = obj["split"]
        prompt = source_info[sid]["prompt"]
        source_text = source_info[sid]["source_info"]
        task = source_info[sid]["task_type"].lower()

        if task not in ["qa"]:
            continue   # 丟掉其他類型

        bucket = stats[task][split]

        # 計算數量
        bucket["count"] += 1

        # 記錄文字長度（字元數）
        bucket["prompt_len"].append(len(prompt))
        bucket["source_len"].append(len(source_text))
        bucket["resp_len"].append(len(obj["response"]))

        # hallucination label
        if "labels" in obj and len(obj["labels"]) > 0:
            bucket["has_label"] += 1


# =====================================
#   印最終統計結果
# =====================================

def print_stat(task, split, st):
    print(f"\n========== {task.upper()} — {split.upper()} ==========")
    print(f"Count: {st['count']}")
    print(f"With hallucination labels: {st['has_label']}")
    print(f"Prompt length: max={max(st['prompt_len'])}, avg={np.mean(st['prompt_len']):.1f}, p95={np.percentile(st['prompt_len'],95)}")
    print(f"Source length: max={max(st['source_len'])}, avg={np.mean(st['source_len']):.1f}, p95={np.percentile(st['source_len'],95)}")
    print(f"Response length: max={max(st['resp_len'])}, avg={np.mean(st['resp_len']):.1f}, p95={np.percentile(st['resp_len'],95)}")


for task in ["qa"]:
    for split in ["train", "test"]:
        print_stat(task, split, stats[task][split])

In [ ]:
# =========================================================
# QA 混合策略 chunk（句子切 + 合併 + 邊界微調 + 重疊）
# =========================================================
def qa_hybrid_chunks(text, max_chunk_size=300, overlap=80):
    sentences = nltk.sent_tokenize(text)
    spans = []
    idx = 0

    # 先取得句子起訖
    sent_spans = []
    for s in sentences:
        start = text.index(s, idx)
        end = start + len(s)
        sent_spans.append((start, end, s))
        idx = end

    # 合併句子直到 max_chunk_size
    chunks = []
    cur_start = None
    cur_end = None
    cur_len = 0

    safe_chars = {'.', ',', ';', '!', '?', ' ', '\n', '。'}

    for start, end, s in sent_spans:
        if cur_start is None:
            cur_start = start
            cur_end = end
            cur_len = end - start
            continue

        if cur_len + (end - start) <= max_chunk_size:
            cur_end = end
            cur_len = cur_end - cur_start
        else:
            # 做安全邊界微調（Avoid cutting inside words）
            adj_end = cur_end
            while adj_end < len(text) and text[adj_end - 1] not in safe_chars:
                adj_end -= 1
                if adj_end <= cur_start + 20:
                    adj_end = cur_end
                    break

            chunks.append([cur_start, adj_end])

            # overlap
            next_start = adj_end - overlap
            if next_start < 0:
                next_start = 0

            # 微調 overlap 的 start，不切到單字
            while next_start < len(text) and text[next_start] not in safe_chars:
                next_start += 1

            cur_start = next_start
            cur_end = end
            cur_len = cur_end - cur_start

    # 補最後一段
    if cur_start is not None:
        chunks.append([cur_start, cur_end])

    return chunks

In [ ]:
# ---------------------------------------------------------
# 讀取 SOURCE_FILE
# ---------------------------------------------------------
source_dict = {}
with open(SOURCE_FILE, "r") as f:
    for line in f:
        obj = json.loads(line)
        source_dict[obj["source_id"]] = obj

In [ ]:
# ---------------------------------------------------------
# 開新檔案準備寫入
# ---------------------------------------------------------
src_out = open(OUTPUT_SOURCE_FILE, "w")
rsp_out = open(OUTPUT_RESPONSE_FILE, "w")

In [ ]:
# ---------------------------------------------------------
# 處理 SOURCE_FILE: 產生 prompt_spans
# ---------------------------------------------------------
print("Processing SOURCE_FILE...")
one_sample_qa = None
kk = 1

for sid, obj in tqdm(source_dict.items()):
    task = obj["task_type"].lower()
    prompt = obj["prompt"]

    if task == "qa":
        spans = qa_hybrid_chunks(prompt, max_chunk_size=500, overlap=120)
        if (one_sample_qa is None) and (kk > 3):
            one_sample_qa = (prompt, spans)
        kk = kk + 1

    else:
        continue

    obj["prompt_spans"] = spans
    src_out.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [ ]:
# ---------------------------------------------------------
# 處理 RESPONSE_FILE: 產生 response_spans
# ---------------------------------------------------------
print("Processing RESPONSE_FILE...")
with open(RESPONSE_FILE, "r") as f:
    for line in tqdm(f):
        obj = json.loads(line)
        sid = obj["source_id"]

        if sid not in source_dict:
            continue

        task = source_dict[sid]["task_type"].lower()
        text = obj["response"]

        if task == "qa":
            spans = qa_hybrid_chunks(text, max_chunk_size=350, overlap=80)
        else:
            continue

        obj["response_spans"] = spans
        rsp_out.write(json.dumps(obj, ensure_ascii=False) + "\n")

src_out.close()
rsp_out.close()

print("\nDone! Two new files generated:")
print(" -", OUTPUT_SOURCE_FILE)
print(" -", OUTPUT_RESPONSE_FILE)

In [ ]:
# ---------------------------------------------------------
# 印出示例： QA
# ---------------------------------------------------------

print("\n==============================")
print(" QA 混合策略切分範例")
print("==============================")
if one_sample_qa:
    text, spans = one_sample_qa
    for i, (s, e) in enumerate(spans):
        print(f"\n[Chunk {i}] ({s} ~ {e})")
        print(text[s:e])